In [6]:
# using google gemini

# https://ai.google.dev/gemini-api/docs/available-regions#unpaid-tier-unavailable

# Create the free api key here:
# https://aistudio.google.com/app/apikey

# Install the required library
!pip install -q -U google-genai

# install dotenv
!pip install -q python-dotenv

In [7]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API")
client = genai.Client(api_key=api_key)

print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

print("-------\n")
print("List of models that support embedContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "embedContent":
            print(m.name)

List of models that support generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
-------

List of models that support embedContent:

models/gemini-embedding-001


In [14]:
# read all files in the articles directory
articles = {}
for filename in os.listdir("articles"):
    with open(os.path.join("articles", filename), "r") as f:
        articles[filename] = f.read()
        
print(f"Read {len(articles)} articles.")

Read 10 articles.


In [22]:
import time
from google.genai import types

# Tell the model to be a helpful assistant that summarizes articles. 
# Provide how you want the summary to be formatted.
instructions = """
You are a helpful assistant that summarizes articles.
Summarize the articles to a person who has not read them nor knows anything about them or the topic.
Output constraints:
- Output 5 bullet points.
- Each bullet point should be no more than 1-2 sentences long.
- Keep the sentences simple and easy to understand, avoiding technical jargon.
- One sentence should be no more than 20 words.
- Neutral tone, no opinions or assumptions.
- Output just the bullet points, no other text.
"""

def ask(text):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0.2,
        ),
        contents=text
    )
    return response.text

summaries = {}
for filename, content in articles.items():
    print(f"Summarizing {filename}...")
    retries = 3
    timeout = 10
    while retries > 0:
        try:
            summaries[filename] = ask(content)
            print(f"Summarized {filename}.\n")
            break 
        except Exception as e:
            retries -= 1
            if retries > 0:
                print(f"Error, retrying in {timeout} seconds... ({retries} attempts left)")
                time.sleep(timeout)
            else:
                print(f"Failed after retries: {filename}")
    
    

Summarizing wordpress.txt...
Summarized wordpress.txt.

Summarizing OSM.txt...
Summarized OSM.txt.

Summarizing wayback_machine.txt...
Summarized wayback_machine.txt.

Summarizing wikipedia.txt...
Summarized wikipedia.txt.

Summarizing chatoyancy.txt...
Summarized chatoyancy.txt.

Summarizing youtube.txt...
Summarized youtube.txt.

Summarizing 8086.txt...
Summarized 8086.txt.

Summarizing c_lang.txt...
Summarized c_lang.txt.

Summarizing annas_arch.txt...
Summarized annas_arch.txt.

Summarizing chatgpt.txt...
Error, retrying in 10 seconds... (2 attempts left)
Error, retrying in 10 seconds... (1 attempts left)
Failed after retries: chatgpt.txt


In [23]:
# print all summaries
for filename, summary in summaries.items():
    print(f"Summary of {filename}:\n{summary}\n")
    
# save summaries as json file
import json
with open("summaries.json", "w") as f:
    json.dump(summaries, f, indent=4)

Summary of wordpress.txt:
*   WordPress is a system for managing website content. It began for blogs but now builds many types of websites.
*   It is free, open-source software built using the PHP programming language. WordPress needs to be installed on a web server to function.
*   Themes let users change a website's appearance. They also modify how the site works without altering its core content.
*   Plugins add new features or change existing ones. Thousands of plugins are available to tailor a site to specific needs.
*   WordPress was created in 2003 by two developers. It is owned by the WordPress Foundation.

Summary of OSM.txt:
*   OpenStreetMap (OSM) is a map database. Volunteers maintain it through open collaboration.
*   Steve Coast founded OpenStreetMap in 2004. He wanted map data to be freely accessible.
*   Volunteers collect map data from surveys. They also trace from aerial images.
*   The OpenStreetMap Foundation hosts the database. This non-profit organization relies o